# Exercise 3 — Predictive Pipeline (Student)

In [ ]:
# ============================================================
# Part 0 — Colab Environment
# ============================================================
!pip -q install sentence-transformers transformers librosa soundfile gradio gensim

import os, math, warnings, json, random
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, mean_absolute_error, mean_squared_error

In [ ]:
# ============================================================
# Part 1 — DATASET ADAPTER
# แก้ไข "เฉพาะส่วนนี้" เมื่อนำโปรแกรมไปใช้กับ Dataset อื่น
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/Colab Notebooks")
DATA_ROOT = BASE / "Dataset_Chapter3_4"

# Dataset_Chapter3_4 ที่สร้างไว้มี data/ เป็น subfolder
DATA_DIR = DATA_ROOT / "data" if (DATA_ROOT / "data").exists() else DATA_ROOT

# ---------- Dataset-specific file names ----------
ITEM_FILE = DATA_DIR / "metadata.csv"
RATING_FILE = DATA_DIR / "ratings.csv"

# ---------- Dataset-specific column aliases ----------
ITEM_ALIASES = {
    "movie_id":"item_id", "id":"item_id",
    "movie_title":"title", "name":"title",
    "genres":"genre", "description":"synopsis",
    "plot":"synopsis", "summary":"synopsis"
}
RATING_ALIASES = {
    "uid":"user_id", "user":"user_id",
    "iid":"item_id", "movie_id":"item_id",
    "score":"rating"
}

# ---------- Dataset-specific media folders ----------
MEDIA_DIRS = {
    "image": DATA_DIR / "images",
    "audio": DATA_DIR / "audio",
    "video": DATA_DIR / "video",
}
MEDIA_EXTS = {
    "image":[".png",".jpg",".jpeg",".webp"],
    "audio":[".wav",".mp3",".flac",".m4a"],
    "video":[".mp4",".mov",".avi",".mkv"],
}

def _rename_if_needed(df, aliases):
    df=df.copy()
    for old,new in aliases.items():
        if old in df.columns and new not in df.columns:
            df=df.rename(columns={old:new})
    return df

def _find_media(folder, item_id, exts):
    for ext in exts:
        p=folder/f"{item_id}{ext}"
        if p.exists():
            return str(p)
    return None

def load_dataset():
    items=_rename_if_needed(pd.read_csv(ITEM_FILE),ITEM_ALIASES)
    ratings=_rename_if_needed(pd.read_csv(RATING_FILE),RATING_ALIASES)

    need_i=["item_id","title","genre","synopsis"]
    need_r=["user_id","item_id","rating"]
    miss_i=[c for c in need_i if c not in items.columns]
    miss_r=[c for c in need_r if c not in ratings.columns]
    if miss_i or miss_r:
        raise ValueError(f"Missing item columns={miss_i}, rating columns={miss_r}")

    items["item_id"]=items["item_id"].astype(str)
    ratings["item_id"]=ratings["item_id"].astype(str)
    ratings["user_id"]=ratings["user_id"].astype(str)
    ratings["rating"]=pd.to_numeric(ratings["rating"])

    for c in ["title","genre","synopsis"]:
        items[c]=items[c].fillna("").astype(str)

    # Text input used by Chapter 3 / text modality
    items["text_content"]=(items["genre"]+" "+items["synopsis"]).str.strip()

    for modality in ["image","audio","video"]:
        items[f"{modality}_path"]=items["item_id"].apply(
            lambda x: _find_media(MEDIA_DIRS[modality],x,MEDIA_EXTS[modality])
        )

    return items,ratings

items,ratings=load_dataset()
ITEM_INDEX={iid:i for i,iid in enumerate(items["item_id"])}

print("DATA_ROOT =",DATA_ROOT)
print("Items =",len(items),"Users =",ratings["user_id"].nunique(),"Ratings =",len(ratings))
display(items[["item_id","title","genre","image_path","audio_path","video_path"]].head())

In [ ]:
# ============================================================
# Part 2 — REUSABLE CORE
# ส่วนนี้ไม่ผูกกับ MovieMini-12 โดยตรง
# ขอเพียง items / ratings อยู่ใน canonical schema
# ============================================================
LIKED_THRESHOLD=4.0
DISLIKED_THRESHOLD=2.0
N_USERS=max(1,ratings["user_id"].nunique())
POPULARITY=ratings.groupby("item_id")["user_id"].nunique().to_dict()

def l2_rows(X):
    return normalize(np.asarray(X,dtype=float))

def transform_vectors(X,mode="L2",pca_dim=16):
    X=np.asarray(X,dtype=float)
    if mode=="Raw":
        return X.copy()
    if mode=="L2":
        return normalize(X)
    if mode=="Center+L2":
        return normalize(X-X.mean(axis=0,keepdims=True))
    if mode=="Standard+L2":
        return normalize(StandardScaler().fit_transform(X))
    if mode=="PCA+L2":
        d=min(pca_dim, max(1,X.shape[0]-1), X.shape[1])
        return normalize(PCA(n_components=d,random_state=42).fit_transform(X))
    raise ValueError(mode)

def _unit(v):
    v=np.asarray(v,dtype=float)
    n=np.linalg.norm(v)
    return v if n==0 else v/n

# User Profile 1: Positive-only
def profile_positive_only(uid,X,rdf):
    r=rdf[rdf["user_id"].astype(str)==str(uid)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    idx=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    if not idx:
        idx=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
    return _unit(X[idx].mean(axis=0)) if idx else np.zeros(X.shape[1])

# User Profile 2: Rating-centered
def profile_rating_centered(uid,X,rdf):
    r=rdf[rdf["user_id"].astype(str)==str(uid)].copy()
    r=r[r["item_id"].astype(str).isin(ITEM_INDEX)]
    if r.empty: return np.zeros(X.shape[1])
    idx=np.array([ITEM_INDEX[str(i)] for i in r["item_id"]])
    w=r["rating"].astype(float).values-float(r["rating"].mean())
    den=np.abs(w).sum()
    p=X[idx].mean(axis=0) if den<=1e-12 else (X[idx]*w[:,None]).sum(axis=0)/den
    return _unit(p)

# User Profile 3: Rocchio
def profile_rocchio(uid,X,rdf,alpha=1.0,beta=1.0):
    r=rdf[rdf["user_id"].astype(str)==str(uid)]
    pos=r[r["rating"]>=LIKED_THRESHOLD]
    neg=r[r["rating"]<=DISLIKED_THRESHOLD]
    pi=[ITEM_INDEX[str(i)] for i in pos["item_id"] if str(i) in ITEM_INDEX]
    ni=[ITEM_INDEX[str(i)] for i in neg["item_id"] if str(i) in ITEM_INDEX]
    vp=X[pi].mean(axis=0) if pi else np.zeros(X.shape[1])
    vn=X[ni].mean(axis=0) if ni else np.zeros(X.shape[1])
    p=alpha*vp-beta*vn
    if np.linalg.norm(p)==0:
        hi=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
        p=X[hi].mean(axis=0) if hi else p
    return _unit(p)

PROFILE_METHODS=["Positive-only","Rating-centered","Rocchio"]

def make_profile(uid,X,rdf,method="Rating-centered",alpha=1.0,beta=1.0):
    if method=="Positive-only": return profile_positive_only(uid,X,rdf)
    if method=="Rating-centered": return profile_rating_centered(uid,X,rdf)
    if method=="Rocchio": return profile_rocchio(uid,X,rdf,alpha,beta)
    raise ValueError(method)

def rank_by_vector(uid,X,rdf,method="Rating-centered",candidates=None,alpha=1.0,beta=1.0):
    p=make_profile(uid,X,rdf,method,alpha,beta)
    s=cosine_similarity(p.reshape(1,-1),X).ravel()
    seen=set(rdf.loc[rdf["user_id"].astype(str)==str(uid),"item_id"].astype(str))
    cand=candidates if candidates is not None else [iid for iid in items["item_id"] if iid not in seen]
    ranked=sorted(cand,key=lambda iid:s[ITEM_INDEX[iid]],reverse=True)
    return ranked,s,p

# ---------------- Accuracy / Ranking metrics ----------------
def precision_at_k(recs,rel,k): return len(set(recs[:k])&set(rel))/k if k else 0.0
def recall_at_k(recs,rel,k):
    rel=set(rel); return len(set(recs[:k])&rel)/len(rel) if rel else 0.0
def f1_at_k(recs,rel,k):
    p=precision_at_k(recs,rel,k); r=recall_at_k(recs,rel,k)
    return 0.0 if p+r==0 else 2*p*r/(p+r)
def hr_at_k(recs,rel,k): return float(bool(set(recs[:k])&set(rel)))
def reciprocal_rank(recs,rel):
    rel=set(rel)
    for i,x in enumerate(recs,1):
        if x in rel:return 1/i
    return 0.0
def ap_at_k(recs,rel,k):
    rel=set(rel); hit=0;s=0
    for i,x in enumerate(recs[:k],1):
        if x in rel: hit+=1;s+=hit/i
    return s/min(len(rel),k) if rel else 0.0
def ndcg_at_k(recs,rel,k):
    rel=set(rel)
    dcg=sum(1/math.log2(i+2) for i,x in enumerate(recs[:k]) if x in rel)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return dcg/idcg if idcg else 0.0

# ---------------- Beyond-Accuracy metrics ----------------
def ild(recs,X):
    idx=[ITEM_INDEX[i] for i in recs if i in ITEM_INDEX]
    if len(idx)<2:return 0.0
    D=1-cosine_similarity(X[idx])
    v=D[np.triu_indices_from(D,k=1)]
    return float(v.mean()) if len(v) else 0.0

def novelty(recs):
    vals=[-math.log2(max(POPULARITY.get(i,1)/N_USERS,1e-12)) for i in recs]
    return float(np.mean(vals)) if vals else 0.0

def calibration_genre(uid,recs,rdf):
    hist=rdf[rdf["user_id"].astype(str)==str(uid)]
    def gs(ids):
        out=[]
        for iid in ids:
            if iid in ITEM_INDEX:
                out += [x.strip() for x in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if x.strip()]
        return out
    a=gs(hist["item_id"].astype(str).tolist()); b=gs(recs)
    if not a or not b:return np.nan
    cats=sorted(set(a)|set(b))
    pa=np.array([a.count(c) for c in cats],float); pa/=pa.sum()
    pb=np.array([b.count(c) for c in cats],float); pb/=pb.sum()
    return float(1-0.5*np.abs(pa-pb).sum())

def coverage(all_recs): return len(set(all_recs))/len(items) if len(items) else 0.0

def gini(all_recs):
    c=pd.Series(all_recs).value_counts()
    x=np.sort(np.array([c.get(i,0) for i in items["item_id"]],float))
    if x.sum()==0:return 0.0
    n=len(x)
    return float(2*np.sum(np.arange(1,n+1)*x)/(n*x.sum())-(n+1)/n)

# ---------------- Offline splits ----------------
def split_loo(rdf):
    folds=[]
    for uid,g in rdf.groupby("user_id"):
        p=g[g["rating"]>=LIKED_THRESHOLD]
        if p.empty: continue
        hold=p.sort_values("rating").iloc[-1]
        folds.append((str(uid),rdf.drop(index=hold.name),str(hold["item_id"])))
    return folds

def split_random_holdout(rdf,seed=42):
    rng=np.random.default_rng(seed); folds=[]
    for uid,g in rdf.groupby("user_id"):
        p=g[g["rating"]>=LIKED_THRESHOLD]
        if p.empty:continue
        hold=p.iloc[int(rng.integers(0,len(p)))]
        folds.append((str(uid),rdf.drop(index=hold.name),str(hold["item_id"])))
    return folds

def split_temporal(rdf):
    if "timestamp" not in rdf.columns:return None
    folds=[]
    for uid,g in rdf.groupby("user_id"):
        p=g[g["rating"]>=LIKED_THRESHOLD].sort_values("timestamp")
        if p.empty:continue
        hold=p.iloc[-1]
        folds.append((str(uid),rdf.drop(index=hold.name),str(hold["item_id"])))
    return folds

def evaluate_vector_space(X,label="",profile_method="Rating-centered",k=3,folds=None,
                          candidate_mode="Full Catalog",n_neg=20,alpha=1.0,beta=1.0,seed=42):
    folds=folds or split_loo(ratings)
    rng=np.random.default_rng(seed)
    rows=[];all_recs=[]
    for uid,train,heldout in folds:
        seen=set(train.loc[train["user_id"].astype(str)==uid,"item_id"].astype(str))
        catalog=[i for i in items["item_id"] if i not in seen]
        if heldout not in catalog: catalog.append(heldout)
        if candidate_mode=="Sampled Negatives":
            neg=[i for i in catalog if i!=heldout]
            take=min(n_neg,len(neg))
            cand=[heldout]+(list(rng.choice(neg,take,replace=False)) if take else [])
        else:
            cand=catalog
        ranked,s,_=rank_by_vector(uid,X,train,profile_method,cand,alpha,beta)
        top=ranked[:k];all_recs+=top
        y=np.array([1 if i==heldout else 0 for i in ranked])
        ys=np.array([s[ITEM_INDEX[i]] for i in ranked])
        auc=roc_auc_score(y,ys) if len(np.unique(y))==2 else np.nan
        rows.append({
            "user_id":uid,"heldout":heldout,
            f"Precision@{k}":precision_at_k(top,[heldout],k),
            f"Recall@{k}":recall_at_k(top,[heldout],k),
            f"F1@{k}":f1_at_k(top,[heldout],k),
            f"HR@{k}":hr_at_k(top,[heldout],k),
            "MRR":reciprocal_rank(ranked,[heldout]),
            f"MAP@{k}":ap_at_k(top,[heldout],k),
            f"nDCG@{k}":ndcg_at_k(top,[heldout],k),
            "AUC":auc,
            "ILD":ild(top,X),"Novelty":novelty(top),
            "Calibration":calibration_genre(uid,top,train),
        })
    d=pd.DataFrame(rows)
    s=d.select_dtypes(include=np.number).mean().to_dict() if len(d) else {}
    s["Coverage"]=coverage(all_recs);s["Gini"]=gini(all_recs);s["Label"]=label
    return pd.Series(s),d

def quick_eval(X,label):
    rows=[]
    for p in PROFILE_METHODS:
        s,_=evaluate_vector_space(X,label,p,k=3)
        rows.append({"Vector":label,"Profile":p,**s.to_dict()})
    cols=["Vector","Profile","HR@3","MRR","nDCG@3","Coverage","Gini","ILD","Novelty","Calibration"]
    return pd.DataFrame(rows)[[c for c in cols if c in pd.DataFrame(rows).columns]]

## ข้อ 1
ใน Lab 3 เปลี่ยน `PROFILE_FOR_MODEL` จาก Rating-centered เป็น Positive-only และ Rocchio  
**คำถาม:** ต้อง rebuild pair features หรือไม่ เพราะเหตุใด?

## ข้อ 2 — ทดลอง Logistic Regression C = 0.1, 1, 10 โดยคง train/test เดิม

## ข้อ 3 — อธิบายความแตกต่างระหว่าง AUC ของ classifier กับ MAE/RMSE ของ rating regressor

## ข้อ 4 — ระบุ data leakage ที่อาจเกิดใน pipeline อย่างน้อย 3 จุดและวิธีป้องกัน